# Chapter 6: Machine Learning - The Inductive View

```{admonition} Learning Objectives
:class: tip
- Understand supervised vs unsupervised learning
- Master decision tree construction (ID3, C4.5)
- Apply entropy and information gain
- Implement pruning strategies
- Use random forests for ensemble learning
- Apply k-Nearest Neighbors classification
- Understand Naive Bayes classifier
- Evaluate models with cross-validation
- Apply train/validation/test splits
```

```{epigraph}
In God we trust. All others must bring data.

-- W. Edwards Deming
```

## 6.1 Introduction

Machine learning represents the **inductive** approach to artificial intelligence, where systems learn patterns from data rather than relying on hand-coded rules. This contrasts with the **deductive** approach (propositional and first-order logic) covered in previous chapters.

### Deductive vs Inductive Learning

**Deductive Reasoning** (Chapters 4-5):
- Starts with general rules and axioms
- Derives specific conclusions
- Example: All mammals give birth to live babies; cats are mammals → cats give birth to live babies
- Limitations: Requires domain experts, cannot discover new patterns

**Inductive Learning** (This chapter):
- Starts with specific examples (data)
- Discovers general patterns
- Example: Given examples of cancer patients and their features → learn diagnostic rules
- Advantages: Discovers novel patterns, scales with data

### Types of Machine Learning

**1. Supervised Learning**
- Training data includes correct labels
- Goal: Learn mapping from features to labels
- Examples: Classification (spam detection), Regression (price prediction)

**2. Unsupervised Learning** (Chapter 9)
- Training data has no labels
- Goal: Discover hidden structure
- Examples: Clustering (customer segmentation), Dimensionality reduction

**3. Reinforcement Learning** (Chapter 10)
- Agent learns through trial and error
- Goal: Maximize cumulative reward
- Examples: Game playing, robotics

### Supervised Learning Framework

**Training Set**: $\mathcal{D} = \{(\mathbf{x}_1, y_1), (\mathbf{x}_2, y_2), ..., (\mathbf{x}_n, y_n)\}$

where:
- $\mathbf{x}_i \in \mathbb{R}^d$ is a feature vector
- $y_i$ is the label (class for classification, real value for regression)

**Goal**: Learn hypothesis $h: \mathcal{X} \rightarrow \mathcal{Y}$ that generalizes to unseen data

**Classification vs Regression**:
- **Classification**: $y \in \{1, 2, ..., k\}$ (discrete labels)
- **Regression**: $y \in \mathbb{R}$ (continuous values)

## 6.2 Overfitting and Validation

### The Overfitting Problem

**Overfitting** occurs when a model learns noise and random fluctuations in training data rather than the underlying pattern.

**Symptoms**:
- Perfect accuracy on training data
- Poor accuracy on test data
- Model is too complex relative to amount of training data

**Example**: A decision tree grown to full height will achieve 100% training accuracy even on random labels, but will fail on test data.

### Bias-Variance Tradeoff

Total error can be decomposed:

$$\text{Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

**Bias**: Error from incorrect assumptions (underfitting)
- High bias: Model too simple, cannot capture pattern
- Example: Linear model for nonlinear relationship

**Variance**: Error from sensitivity to training data fluctuations (overfitting)
- High variance: Model too complex, fits noise
- Example: Decision tree with one leaf per training point

**Goal**: Find the sweet spot that minimizes total error

### Train/Validation/Test Split

Data must be divided into three parts:

**1. Training Data (50-70%)**:
- Build models
- Can be used multiple times

**2. Validation Data (15-25%)**:
- Tune hyperparameters
- Select best model
- Prevents overfitting during model selection

**3. Test Data (15-25%)**:
- Final evaluation
- **Can only be used ONCE**
- Never look at test data during development

**Critical Rule**: Never mix test data into training or validation!

## 6.3 Decision Trees

Decision trees partition the feature space hierarchically using split conditions to create regions dominated by specific classes.

### 6.3.1 Basic Concepts

**Structure**:
- **Internal nodes**: Split conditions (e.g., Age < 30)
- **Edges**: Outcomes of splits
- **Leaf nodes**: Class predictions

**Types of Splits**:

1. **Univariate split**: Uses single attribute
   - Example: Age < 50

2. **Multivariate split**: Uses multiple attributes
   - Example: Age < 50 AND Salary > 60000
   - More powerful but harder to interpret

**Handling Different Attribute Types**:

| Attribute Type | Split Method |
|----------------|--------------|
| Binary | Two branches |
| Categorical (r values) | r-way split or binary grouping |
| Numeric | Binary threshold: $x \leq a$ |

### 6.3.2 Entropy and Information Gain

**Entropy** measures impurity or uncertainty in a set.

For a set $S$ with class distribution $(p_1, p_2, ..., p_k)$:

$$E(S) = -\sum_{j=1}^{k} p_j \log_2(p_j)$$

where $p_j$ is the fraction of examples in class $j$.

**Properties**:
- $E(S) = 0$ when all examples same class (perfectly pure)
- $E(S)$ is maximum when classes equally distributed
- For binary classification: $E(S) = -p\log_2(p) - (1-p)\log_2(1-p)$

**Example**: 
- 9 positive, 5 negative examples
- $p_+ = 9/14 = 0.643$, $p_- = 5/14 = 0.357$
- $E(S) = -0.643\log_2(0.643) - 0.357\log_2(0.357) = 0.940$

**Information Gain** measures entropy reduction after a split:

$$\text{Gain}(S, A) = E(S) - \sum_{v \in \text{Values}(A)} \frac{|S_v|}{|S|} E(S_v)$$

where $S_v$ is the subset of $S$ for which attribute $A$ has value $v$.

**Goal**: Choose attribute with highest information gain for splitting.

### 6.3.3 Gini Index

**Gini Index** is an alternative impurity measure:

$$G(S) = 1 - \sum_{j=1}^{k} p_j^2$$

For a split into subsets $S_1, ..., S_r$:

$$\text{Gini-Split}(S, S_1, ..., S_r) = \sum_{i=1}^{r} \frac{|S_i|}{|S|} G(S_i)$$

**Properties**:
- $G(S) = 0$ when all examples same class
- Used by CART algorithm
- Computationally faster than entropy

### 6.3.4 Split Criteria Comparison

**1. Error Rate**:
- Fraction of misclassified examples: $1 - p_{\text{majority}}$
- Simple but less sensitive to changes

**2. Entropy** (ID3, C4.5):
- Information-theoretic measure
- More sensitive to probability changes

**3. Gini Index** (CART):
- Similar to entropy
- Slightly faster to compute

| Criterion | Formula | Algorithm |
|-----------|---------|-----------|
| Error Rate | $1 - \max_j p_j$ | Simple trees |
| Entropy | $-\sum_j p_j \log_2 p_j$ | ID3, C4.5 |
| Gini | $1 - \sum_j p_j^2$ | CART |

## 6.4 Decision Tree Algorithms

### 6.4.1 ID3 Algorithm

**ID3** (Iterative Dichotomiser 3) uses information gain to build trees.

```
Algorithm: ID3(Examples, Attributes, Target)

begin
    // Base cases
    if all examples have same class c then
        return leaf node labeled c
    
    if Attributes is empty then
        return leaf node labeled with majority class
    
    // Recursive case
    A ← attribute in Attributes with highest Gain(Examples, A)
    
    Create root node with test attribute A
    
    for each value v of A do
        Examples_v ← subset of Examples where A = v
        
        if Examples_v is empty then
            Add leaf child labeled with majority class in Examples
        else
            Add subtree ID3(Examples_v, Attributes - {A}, Target)
    
    return root node
end
```

**Properties**:
- Uses entropy and information gain
- Grows tree to full height
- Handles categorical attributes
- No pruning in original version

### 6.4.2 C4.5 Algorithm

**C4.5** improves ID3 with:

1. **Gain Ratio**: Normalizes information gain
   $$\text{GainRatio}(S, A) = \frac{\text{Gain}(S, A)}{\text{SplitInfo}(S, A)}$$
   
   where:
   $$\text{SplitInfo}(S, A) = -\sum_{i=1}^{r} \frac{|S_i|}{|S|} \log_2\left(\frac{|S_i|}{|S|}\right)$$
   
   This prevents bias toward attributes with many values.

2. **Handles continuous attributes**: Binary splits $x \leq \theta$

3. **Handles missing values**: Distributes examples probabilistically

4. **Pruning**: Post-pruning to reduce overfitting

### 6.4.3 CART Algorithm

**CART** (Classification and Regression Trees):

- Uses Gini index for classification
- Uses variance reduction for regression
- Always creates binary splits
- Cost-complexity pruning

**For numeric attribute splits**:
1. Sort examples by attribute value
2. Consider each midpoint as potential split
3. Choose split with best Gini index

## 6.5 Tree Construction Algorithm

### Complete Training Process

```
Algorithm: CONSTRUCT-DECISION-TREE(Data D)

begin
    // Step 1: Hold out validation set
    H ← Hold out subset from D
    D' ← D - H
    
    // Step 2: Initialize tree
    T ← Single root node containing D'
    
    // Step 3: Tree Construction Phase
    repeat
        Select eligible leaf node L from T with data set D_L
        
        // Find best split
        (A, threshold) ← BEST-SPLIT(D_L)
        
        // Partition data
        D_left ← {x ∈ D_L : x[A] ≤ threshold}
        D_right ← {x ∈ D_L : x[A] > threshold}
        
        // Create children
        Store split (A, threshold) at L
        Create children for L with D_left and D_right
    
    until no more eligible nodes in T
    
    // Step 4: Tree Pruning Phase
    repeat
        Select untested internal node N in T (bottom-up order)
        
        T_pruned ← T with subtree at N removed
        
        Accuracy_original ← EVALUATE(T, H)
        Accuracy_pruned ← EVALUATE(T_pruned, H)
        
        if Accuracy_pruned ≥ Accuracy_original then
            T ← T_pruned
    
    until all internal nodes tested
    
    // Step 5: Label leaves
    for each leaf node L in T do
        Label L with majority class in L
    
    return T
end

Algorithm: BEST-SPLIT(Data D)

begin
    best_gain ← -∞
    best_attribute ← null
    best_threshold ← null
    
    for each attribute A in D do
        if A is categorical then
            for each value v of A do
                gain ← INFORMATION-GAIN(D, A, v)
                if gain > best_gain then
                    best_gain ← gain
                    best_attribute ← A
                    best_threshold ← v
        
        else  // A is numeric
            values ← SORT(D[A])
            for each midpoint θ between consecutive values do
                gain ← INFORMATION-GAIN(D, A, θ)
                if gain > best_gain then
                    best_gain ← gain
                    best_attribute ← A
                    best_threshold → θ
    
    return (best_attribute, best_threshold)
end
```

**Stopping Criteria** (node eligibility):
1. All examples in node have same label
2. No attributes left to split on
3. Node contains fewer than min_samples examples
4. Maximum tree depth reached
5. Information gain below threshold

**Complexity**:
- Building tree: $O(n \cdot d \cdot \log n)$ for $n$ examples, $d$ attributes
- Prediction: $O(\text{depth})$ per example

## 6.6 Pruning Strategies

Pruning reduces overfitting by simplifying the tree.

### 6.6.1 Post-Pruning

**Reduced Error Pruning**:
1. Grow tree to full height on training data
2. For each internal node (bottom-up):
   - Try converting node to leaf
   - Evaluate on validation set
   - Keep pruned version if accuracy improves

**Cost-Complexity Pruning** (used by CART):

Define cost-complexity measure:
$$C_\alpha(T) = \text{Error}(T) + \alpha |\text{Leaves}(T)|$$

where $\alpha$ balances error vs tree size.

Algorithm:
1. Generate sequence of trees $T_0, T_1, ..., T_k$ with increasing $\alpha$
2. Evaluate each on validation set
3. Select tree with best validation accuracy

### 6.6.2 Pre-Pruning

Stop growing tree early based on criteria:

1. **Minimum samples per leaf**: Stop if split creates leaf with < min_samples
2. **Maximum depth**: Limit tree depth
3. **Minimum information gain**: Stop if gain < threshold
4. **Statistical significance**: Chi-square test for split significance

**Advantages**: Faster, avoids building large tree
**Disadvantages**: May stop too early, missing good subtrees

### 6.6.3 Pessimistic Error Pruning

Estimate error rate with confidence intervals:

For node with $n$ examples and $e$ errors:
$$\text{PessimisticError} = \frac{e + 0.5}{n}$$

Prune if parent's pessimistic error ≤ sum of children's errors.

## 6.7 Random Forests

**Random Forests** are ensembles of decision trees that reduce overfitting through randomization and averaging.

### 6.7.1 Key Ideas

**1. Bootstrap Aggregating (Bagging)**:
- Create $B$ bootstrap samples from training data
- Train one tree on each sample
- Average predictions (regression) or vote (classification)

**2. Random Feature Selection**:
- At each split, randomly select $m$ features from $d$ total
- Choose best split from these $m$ features only
- Typical: $m = \sqrt{d}$ for classification, $m = d/3$ for regression

**3. No Pruning**:
- Grow each tree to full height
- Averaging compensates for individual tree overfitting

### 6.7.2 Random Forest Algorithm

```
Algorithm: RANDOM-FOREST(Data D, num_trees B, num_features m)

begin
    forest ← []
    
    for i = 1 to B do
        // Bootstrap sample
        D_i ← Sample n examples from D with replacement
        
        // Build tree with random feature selection
        T_i ← BUILD-RANDOMIZED-TREE(D_i, m)
        
        forest.append(T_i)
    
    return forest
end

Algorithm: BUILD-RANDOMIZED-TREE(Data D, num_features m)

begin
    if stopping criterion met then
        return leaf with majority class
    
    // Randomly select m features
    F ← Random sample of m features from all d features
    
    // Find best split using only F
    (A, θ) ← BEST-SPLIT(D, F)
    
    // Split data
    D_left ← {x ∈ D : x[A] ≤ θ}
    D_right ← {x ∈ D : x[A] > θ}
    
    // Recursively build subtrees
    left_child ← BUILD-RANDOMIZED-TREE(D_left, m)
    right_child ← BUILD-RANDOMIZED-TREE(D_right, m)
    
    return node with split (A, θ), left_child, right_child
end

Algorithm: RF-PREDICT(forest, instance x)

begin
    votes ← {}
    
    for each tree T in forest do
        prediction ← T.predict(x)
        votes[prediction] ← votes[prediction] + 1
    
    return class with most votes
end
```

### 6.7.3 Why Random Forests Work

**Variance Reduction**:

For $B$ trees with predictions $f_1(x), ..., f_B(x)$:

Average prediction: $\bar{f}(x) = \frac{1}{B}\sum_{i=1}^{B} f_i(x)$

If trees are independent with variance $\sigma^2$:
$$\text{Var}(\bar{f}) = \frac{\sigma^2}{B}$$

**In practice**: Trees are correlated (correlation $\rho$):
$$\text{Var}(\bar{f}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

**Key insight**: Random feature selection decorrelates trees, reducing $\rho$

### 6.7.4 Properties

**Advantages**:
- Reduces overfitting dramatically
- Handles high-dimensional data
- Robust to outliers and noise
- Provides feature importance estimates
- Parallelizable

**Disadvantages**:
- Less interpretable than single tree
- Slower prediction than single tree
- Memory intensive (stores $B$ trees)

**Typical Parameters**:
- $B = 100$ to $500$ trees
- $m = \sqrt{d}$ for classification
- No pruning (max_depth = None)

## 6.8 k-Nearest Neighbors

**k-Nearest Neighbors (k-NN)** is a lazy learning algorithm that makes predictions based on similar training examples.

### 6.8.1 Algorithm

```
Algorithm: k-NN-CLASSIFY(Training data D, Test instance x, k)

begin
    // Compute distances to all training examples
    distances ← []
    for each (x_i, y_i) in D do
        d ← DISTANCE(x, x_i)
        distances.append((d, y_i))
    
    // Sort by distance
    distances.sort()
    
    // Get k nearest neighbors
    neighbors ← distances[1:k]
    
    // Majority vote
    votes ← {}
    for (d, y) in neighbors do
        votes[y] ← votes[y] + 1
    
    return class with most votes
end
```

### 6.8.2 Distance Metrics

**Euclidean Distance** (most common):
$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{i=1}^{d} (x_i - y_i)^2}$$

**Manhattan Distance**:
$$d(\mathbf{x}, \mathbf{y}) = \sum_{i=1}^{d} |x_i - y_i|$$

**Minkowski Distance** (generalization):
$$d(\mathbf{x}, \mathbf{y}) = \left(\sum_{i=1}^{d} |x_i - y_i|^p\right)^{1/p}$$

**Cosine Similarity** (for text):
$$\text{sim}(\mathbf{x}, \mathbf{y}) = \frac{\mathbf{x} \cdot \mathbf{y}}{||\mathbf{x}|| \cdot ||\mathbf{y}||}$$

### 6.8.3 Choosing k

**Small k** (e.g., k=1):
- High variance, low bias
- Sensitive to noise
- Complex decision boundaries

**Large k**:
- Low variance, higher bias
- Smoother decision boundaries
- May underfit

**Rule of thumb**: $k = \sqrt{n}$ where $n$ is training set size

**Cross-validation** to select k:
```
for k in [1, 3, 5, 7, 9, ...] do
    accuracy ← CROSS-VALIDATE(D, k)
    
best_k ← k with highest accuracy
```

### 6.8.4 Weighted k-NN

Weight neighbors by inverse distance:

$$w_i = \frac{1}{d(\mathbf{x}, \mathbf{x}_i)^2}$$

Prediction: $\hat{y} = \arg\max_c \sum_{i \in \text{neighbors}} w_i \cdot \mathbb{1}(y_i = c)$

**Advantage**: Closer neighbors have more influence

### 6.8.5 Complexity and Properties

**Time Complexity**:
- Training: $O(1)$ (lazy learning - no training phase)
- Prediction: $O(nd)$ where $n$ = training size, $d$ = dimensions

**Space Complexity**: $O(nd)$ (must store all training data)

**Advantages**:
- Simple and intuitive
- No training phase
- Naturally handles multi-class problems
- Can learn complex decision boundaries with enough data

**Disadvantages**:
- Slow prediction for large datasets
- Sensitive to irrelevant features (curse of dimensionality)
- Requires feature scaling
- Memory intensive

**Improvements**:
- KD-trees or ball trees for faster search: $O(d \log n)$
- Dimensionality reduction (PCA)
- Feature selection

## 6.9 Naive Bayes Classifier

**Naive Bayes** is a probabilistic classifier based on Bayes' theorem with independence assumptions.

### 6.9.1 Bayes' Theorem

$$P(c | \mathbf{x}) = \frac{P(\mathbf{x} | c) P(c)}{P(\mathbf{x})}$$

where:
- $P(c | \mathbf{x})$ = posterior probability of class $c$ given features $\mathbf{x}$
- $P(\mathbf{x} | c)$ = likelihood of features given class $c$
- $P(c)$ = prior probability of class $c$
- $P(\mathbf{x})$ = evidence (constant for all classes)

**Classification rule**:
$$\hat{c} = \arg\max_c P(c | \mathbf{x}) = \arg\max_c P(\mathbf{x} | c) P(c)$$

### 6.9.2 Naive Independence Assumption

Assume features are conditionally independent given the class:

$$P(\mathbf{x} | c) = P(x_1, x_2, ..., x_d | c) = \prod_{i=1}^{d} P(x_i | c)$$

This "naive" assumption is often violated but works well in practice!

**Classification**:
$$\hat{c} = \arg\max_c P(c) \prod_{i=1}^{d} P(x_i | c)$$

### 6.9.3 Naive Bayes Algorithm

```
Algorithm: NAIVE-BAYES-TRAIN(Training data D)

begin
    // Estimate prior probabilities
    for each class c do
        P(c) ← (number of examples in class c) / |D|
    
    // Estimate likelihoods
    for each class c do
        for each feature i do
            for each value v of feature i do
                P(x_i = v | c) ← (count of examples with x_i = v and class c) 
                                    / (count of examples in class c)
    
    return P(c) and P(x_i | c) for all c, i
end

Algorithm: NAIVE-BAYES-CLASSIFY(Test instance x, priors, likelihoods)

begin
    best_class ← null
    best_prob ← -∞
    
    for each class c do
        // Compute log probability (to avoid underflow)
        log_prob ← log P(c)
        
        for i = 1 to d do
            log_prob ← log_prob + log P(x_i | c)
        
        if log_prob > best_prob then
            best_prob ← log_prob
            best_class ← c
    
    return best_class
end
```

### 6.9.4 Handling Different Feature Types

**Categorical Features**:
$$P(x_i = v | c) = \frac{\text{count}(x_i = v, c)}{\text{count}(c)}$$

**Continuous Features** (Gaussian Naive Bayes):

Assume $P(x_i | c) \sim \mathcal{N}(\mu_{ic}, \sigma_{ic}^2)$

$$P(x_i | c) = \frac{1}{\sqrt{2\pi\sigma_{ic}^2}} \exp\left(-\frac{(x_i - \mu_{ic})^2}{2\sigma_{ic}^2}\right)$$

Estimate $\mu_{ic}$ and $\sigma_{ic}^2$ from training data.

### 6.9.5 Laplace Smoothing

To handle zero probabilities (unseen feature values):

$$P(x_i = v | c) = \frac{\text{count}(x_i = v, c) + \alpha}{\text{count}(c) + \alpha|V_i|}$$

where $|V_i|$ is number of possible values for feature $i$, and $\alpha$ (typically 1) is smoothing parameter.

**Without smoothing**: $P(x_i = v | c) = 0$ for unseen $v$ makes entire product 0

**With smoothing**: Every value gets small non-zero probability

### 6.9.6 Properties

**Advantages**:
- Fast training and prediction: $O(nd)$
- Works well with high-dimensional data
- Requires small amount of training data
- Handles missing values naturally
- Probabilistic predictions

**Disadvantages**:
- Independence assumption often violated
- Poor probability estimates (but good classifications!)
- Sensitive to irrelevant features

**Applications**:
- Text classification (spam detection, sentiment analysis)
- Medical diagnosis
- Real-time prediction (due to speed)

## 6.10 Model Evaluation

### 6.10.1 Hold-Out Method

Split data into training and test sets:

```
Algorithm: HOLD-OUT-EVALUATION(Data D, split_ratio)

begin
    n ← |D|
    n_train ← ⌊split_ratio × n⌋
    
    // Shuffle and split
    SHUFFLE(D)
    D_train ← D[1:n_train]
    D_test ← D[n_train+1:n]
    
    // Train model
    model ← TRAIN(D_train)
    
    // Evaluate
    accuracy ← EVALUATE(model, D_test)
    
    return accuracy
end
```

**Typical splits**:
- 70/30 (train/test)
- 80/20
- 50/25/25 (train/validation/test)

**Pros**: Fast, simple
**Cons**: High variance, wastes data

### 6.10.2 k-Fold Cross-Validation

```
Algorithm: K-FOLD-CROSS-VALIDATION(Data D, k)

begin
    // Divide data into k equal folds
    SHUFFLE(D)
    folds ← SPLIT(D, k)
    
    accuracies ← []
    
    for i = 1 to k do
        // Use fold i as test set
        D_test ← folds[i]
        D_train ← folds[1:i-1] ∪ folds[i+1:k]
        
        // Train and evaluate
        model ← TRAIN(D_train)
        accuracy ← EVALUATE(model, D_test)
        accuracies.append(accuracy)
    
    // Return average accuracy
    return MEAN(accuracies)
end
```

**Common choices**: $k = 5$ or $k = 10$

**Leave-One-Out** (LOOCV): $k = n$ (each example is a fold)
- Most accurate estimate
- Very expensive for large datasets

**Stratified k-Fold**: Preserve class distribution in each fold

**Pros**: Lower variance, uses all data for both training and testing
**Cons**: $k$ times more expensive than hold-out

### 6.10.3 Evaluation Metrics

**Confusion Matrix** (binary classification):

```
                Predicted
              Pos      Neg
Actual  Pos   TP       FN
        Neg   FP       TN
```

**Accuracy**:
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**Precision** (of positive class):
$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall** (Sensitivity, True Positive Rate):
$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1-Score** (harmonic mean):
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Specificity** (True Negative Rate):
$$\text{Specificity} = \frac{TN}{TN + FP}$$

**When to use what**:
- **Accuracy**: Balanced datasets
- **Precision**: Cost of false positives high (e.g., spam detection)
- **Recall**: Cost of false negatives high (e.g., disease detection)
- **F1**: Imbalanced datasets

### 6.10.4 Multi-Class Metrics

**Macro-Average**: Average metric across classes (treats all classes equally)
$$\text{Macro-F1} = \frac{1}{k}\sum_{i=1}^{k} F_{1,i}$$

**Micro-Average**: Pool all examples, then compute metric
$$\text{Micro-F1} = \frac{\sum_{i} TP_i}{\sum_{i} (TP_i + \frac{1}{2}(FP_i + FN_i))}$$

**Weighted-Average**: Weight by class frequency

## 6.11 Summary

### Key Concepts

1. **Inductive learning**: Learn from examples, not rules
2. **Overfitting**: Model learns noise instead of pattern
3. **Bias-variance tradeoff**: Balance model complexity
4. **Train/validation/test**: Proper data splits are critical

### Algorithms Comparison

| Algorithm | Training | Prediction | Interpretable | Overfitting |
|-----------|----------|------------|---------------|-------------|
| Decision Tree | $O(nd\log n)$ | $O(\text{depth})$ | High | High |
| Random Forest | $O(Bnd\log n)$ | $O(B \cdot \text{depth})$ | Low | Low |
| k-NN | $O(1)$ | $O(nd)$ | Medium | Medium |
| Naive Bayes | $O(nd)$ | $O(d)$ | High | Low |

### When to Use What

**Decision Trees**:
- Need interpretable model
- Mixed feature types
- Non-linear relationships
- Small to medium datasets

**Random Forests**:
- Want high accuracy
- Can sacrifice interpretability
- Large datasets
- High-dimensional data

**k-Nearest Neighbors**:
- No training time constraint
- Small dataset
- Complex decision boundaries
- Similar items naturally clustered

**Naive Bayes**:
- Text classification
- Need speed
- High-dimensional data
- Small training set

### Best Practices

1. **Always split data**: train/val/test
2. **Never touch test data** until final evaluation
3. **Use cross-validation** for reliable estimates
4. **Scale features** for distance-based methods
5. **Handle missing values** appropriately
6. **Check for class imbalance**
7. **Tune hyperparameters** on validation set
8. **Try ensemble methods** for better performance

## 6.12 Implementation

For complete Python implementations, see:

[ch06_ml_intro_implementation.ipynb](ch06_ml_intro_implementation.ipynb)

The implementation notebook includes:

1. **Decision Tree Framework**: ID3, C4.5, CART implementations
2. **Entropy and Information Gain**: Complete calculations
3. **Tree Pruning**: Reduced error pruning and cost-complexity
4. **Random Forest**: Bootstrap aggregating with feature randomization
5. **k-Nearest Neighbors**: With different distance metrics
6. **Naive Bayes**: Gaussian and categorical variants
7. **Evaluation Framework**: Cross-validation, confusion matrix, metrics
8. **Real-World Applications**:
   - Iris flower classification
   - Titanic survival prediction
   - Wine quality classification
   - Spam detection (text classification)

## Further Reading

### Textbooks

- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 6]
- Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. [Chapter 19]
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning*. Springer.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
- Mitchell, T. (1997). *Machine Learning*. McGraw-Hill.

### Classic Papers

- Quinlan, J. R. (1986). Induction of decision trees. *Machine Learning*, 1(1), 81-106.
- Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5-32.
- Cover, T., & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory*, 13(1), 21-27.

### Software Libraries

- **scikit-learn**: [https://scikit-learn.org](https://scikit-learn.org)
- **XGBoost**: Extreme gradient boosting
- **LightGBM**: Fast gradient boosting
- **CatBoost**: Categorical feature boosting